# SIBA PyTorch / Jittor 对齐演示

本 Notebook 只读取项目中已经保存的真实源码审计、数值对齐、训练、推理和指标文件。它不生成替代实验数据，也不修改论文算法。


In [ ]:
import json
from pathlib import Path

PROJECT_ROOT = Path('/root/autodl-tmp/SIBA-Jittor')
RUN_TAG = '20260727_siba_official_protocol'


## 1. 官方文件与 Jittor 文件对应关系


In [ ]:
source_audit = json.loads((PROJECT_ROOT / 'docs/source_audit_final_20260728.json').read_text())
source_audit['comparison']


In [ ]:
official_code = (PROJECT_ROOT / 'official_pytorch/models/SIBA.py').read_text()
jittor_code = (PROJECT_ROOT / 'siba_jittor/models/SIBA.py').read_text()
print('PyTorch SIBA.py lines:', len(official_code.splitlines()))
print('Jittor  SIBA.py lines:', len(jittor_code.splitlines()))


## 2. 数据读取与配对检查


In [ ]:
print(json.dumps(json.loads((PROJECT_ROOT / 'logs/alignment/data_loader_report.json').read_text()), ensure_ascii=False, indent=2))
print(json.dumps(json.loads((PROJECT_ROOT / 'logs/alignment/training_dataset_validation.json').read_text()), ensure_ascii=False, indent=2))


## 3. 前向、损失、梯度与一步更新


In [ ]:
alignment = json.loads((PROJECT_ROOT / 'logs/alignment/alignment_assessment_final_20260728.json').read_text())
alignment['measured'], alignment['conclusions'], alignment['interpretation']


## 4. 官方权重下的 706 张输出对齐


In [ ]:
for dataset in ['MSRS', 'M3FD_2x', 'TNO']:
    path = PROJECT_ROOT / f'results/output_alignment_{RUN_TAG}/{dataset}/summary.json'
    print(dataset)
    print(json.dumps(json.loads(path.read_text()), ensure_ascii=False, indent=2))


## 5. 两框架完整 60 轮训练


In [ ]:
from IPython.display import Image, display

curve = PROJECT_ROOT / f'results/training_analysis_{RUN_TAG}/loss_curve.png'
display(Image(filename=str(curve)))
print((PROJECT_ROOT / f'results/training_analysis_{RUN_TAG}/training_log_summary.json').read_text())


## 6. 指标、速度与显存


In [ ]:
import pandas as pd

display(pd.read_csv(PROJECT_ROOT / f'results/metrics_{RUN_TAG}/metrics_summary.csv'))
display(pd.read_csv(PROJECT_ROOT / f'results/performance_summary_{RUN_TAG}/inference_timing.csv'))
print((PROJECT_ROOT / f'results/performance_summary_{RUN_TAG}/gpu_monitor_summary.json').read_text())


## 7. 复现边界


In [ ]:
print('推理迁移：官方权重下高度一致。')
print('训练功能：完成全量 60 轮并收敛。')
print('严格训练步等价：未通过，不能宣称逐步完全相同。')
print('RoadScene 训练子集：作者未公开具体 200 对名单，不能宣称完全相同。')
